In [1]:
import sys
from pathlib import Path
sys.path.append(f"{Path().absolute().parent}")

In [2]:
import pandas as pd
from notebooks.radp_library import (preprocess_ue_data)
from apps.mobility_robustness_optimization.simple_mro import SimpleMRO
from apps.mobility_robustness_optimization.mro_rl import ReinforcedMRO
from notebooks.radp_library import get_ue_data

In [3]:
simple_ue = pd.read_csv('E:/Repositories/maveric/notebooks/data/sim_data/UE_data_20UE_100ticks.csv')
topology = pd.read_csv('E:/Repositories/maveric/notebooks/data/sim_data/topology.csv')
topology.loc[topology["cell_id"] == "cell_1", "cell_lat"] = -90
topology.loc[topology["cell_id"] == "cell_2", "cell_lat"] = 0
topology.loc[topology["cell_id"] == "cell_3", "cell_lat"] = 90

topology.loc[topology["cell_id"] == "cell_1", "cell_lon"] = -180
topology.loc[topology["cell_id"] == "cell_2", "cell_lon"] = 0
topology.loc[topology["cell_id"] == "cell_3", "cell_lon"] = 180

topology.loc[topology["cell_id"] == "cell_1", "cell_carrier_freq_mhz"] = 2100
topology.loc[topology["cell_id"] == "cell_2", "cell_carrier_freq_mhz"] = 2100
topology.loc[topology["cell_id"] == "cell_3", "cell_carrier_freq_mhz"] = 2100

simple_ue.drop(columns=['mock_ue_id', 'tick'], inplace=True)

In [4]:
params = {
    "ue_tracks_generation": {
            "params": {
                "simulation_duration": 3600,
                "simulation_time_interval_seconds": 0.01,
                "num_ticks": 100,
                "num_batches": 1,
                "ue_class_distribution": {
                    "stationary": {
                        "count": 5,
                        "velocity": 0,
                        "velocity_variance": 1
                    },
                    "pedestrian": {
                        "count": 5,
                        "velocity": 2,
                        "velocity_variance": 1
                    },
                    "cyclist": {
                        "count": 5,
                        "velocity": 5,
                        "velocity_variance": 1
                    },
                    "car": {
                        "count": 5,
                        "velocity": 20,
                        "velocity_variance": 1
                    }
                },
                "lat_lon_boundaries": {
                    "min_lat": -90,
                    "max_lat": 90,
                    "min_lon": -180,
                    "max_lon": 180
                },
                "gauss_markov_params": {
                    "alpha": 0.5,
                    "variance": 0.8,
                    "rng_seed": 42,
                    "lon_x_dims": 100,
                    "lon_y_dims": 100,
                    "// TODO": "Account for supporting the user choosing the anchor_loc and cov_around_anchor.",
                    "// Current implementation": "the UE Tracks generator will not be using these values.",
                    "// anchor_loc": {},
                    "// cov_around_anchor": {}
            }
        }
    }
}

__Start__

In [5]:
input_data = preprocess_ue_data(simple_ue, topology)

### Simple MRO

__init__

In [6]:
mro = SimpleMRO(mobility_model_params = params, topology = topology)

__update 1__

In [7]:
mro.bayesian_digital_twins

{}

In [8]:
mro.train_or_update_rf_twin(input_data)

No Bayesian Digital Twins available for update. Training from scratch.


[2025-05-10 23:09:19,970] INFO:  Iter 1/100 - Loss: 0.784 (delta=inf)
[2025-05-10 23:09:20,109] INFO:  Iter 2/100 - Loss: 0.765 (delta=-0.019081)
[2025-05-10 23:09:20,251] INFO:  Iter 3/100 - Loss: 0.745 (delta=-0.019101)
[2025-05-10 23:09:20,386] INFO:  Iter 4/100 - Loss: 0.726 (delta=-0.019145)
[2025-05-10 23:09:20,524] INFO:  Iter 5/100 - Loss: 0.707 (delta=-0.019228)
[2025-05-10 23:09:20,662] INFO:  Iter 6/100 - Loss: 0.688 (delta=-0.019359)
[2025-05-10 23:09:20,872] INFO:  Iter 7/100 - Loss: 0.668 (delta=-0.019524)
[2025-05-10 23:09:21,027] INFO:  Iter 8/100 - Loss: 0.648 (delta=-0.019723)
[2025-05-10 23:09:21,172] INFO:  Iter 9/100 - Loss: 0.629 (delta=-0.019923)
[2025-05-10 23:09:21,318] INFO:  Iter 10/100 - Loss: 0.608 (delta=-0.020137)
[2025-05-10 23:09:21,463] INFO:  Iter 11/100 - Loss: 0.588 (delta=-0.020355)
[2025-05-10 23:09:21,608] INFO:  Iter 12/100 - Loss: 0.567 (delta=-0.020540)
[2025-05-10 23:09:21,756] INFO:  Iter 13/100 - Loss: 0.547 (delta=-0.020726)
[2025-05-10 23

In [9]:
mro.bayesian_digital_twins

{'cell_1': <radp.digital_twin.rf.bayesian.bayesian_engine.BayesianDigitalTwin at 0x2ad339df9a0>,
 'cell_2': <radp.digital_twin.rf.bayesian.bayesian_engine.BayesianDigitalTwin at 0x2ad0ff23640>,
 'cell_3': <radp.digital_twin.rf.bayesian.bayesian_engine.BayesianDigitalTwin at 0x2ad33b64c70>}

__solve__

In [10]:
mro.solve(n_epochs=3)

Epoch  Hyst           TTT    MRO Metric  
-----------------------------------------
0      1.4165746239   44     99.550000   
1      0.0833020791   30     99.650000   
2      2.3455049003   73     99.350000   

Optimized Hyst: 0.01,
            Optimized TTT: 5


(0.01, 5)

In [11]:
new_ue = get_ue_data(params)
new_ue.rename(columns={"lon": "longitude", "lat": "latitude"}, inplace=True)
new_data = preprocess_ue_data(new_ue, topology)

__update 2__

In [12]:
mro.train_or_update_rf_twin(new_data) # FIXME: Buggy

Updating existing Bayesian Digital Twins with new data.
  → cell 'cell_1' has 300 unique rows after dedupe/subsample
  → cell 'cell_2' has 300 unique rows after dedupe/subsample
  → cell 'cell_3' has 300 unique rows after dedupe/subsample


In [13]:
mro.bayesian_digital_twins

{'cell_1': <radp.digital_twin.rf.bayesian.bayesian_engine.BayesianDigitalTwin at 0x2ad339df9a0>,
 'cell_2': <radp.digital_twin.rf.bayesian.bayesian_engine.BayesianDigitalTwin at 0x2ad0ff23640>,
 'cell_3': <radp.digital_twin.rf.bayesian.bayesian_engine.BayesianDigitalTwin at 0x2ad33b64c70>}

In [14]:
mro.solve(n_epochs=3)

Epoch  Hyst           TTT    MRO Metric  
-----------------------------------------
0      0.6825171358   10     100.000000  
1      0.3724863024   62     99.850000   
2      1.0087849074   13     100.000000  

Optimized Hyst: 0.6825171358017518,
            Optimized TTT: 10


(0.6825171358017518, 10)

### RL MRO

In [ ]:
rl_mro = ReinforcedMRO(mobility_model_params = params, topology = topology)

In [ ]:
input_data = preprocess_ue_data(simple_ue, topology)

In [ ]:
rl_mro.train_or_update_rf_twin(input_data)

In [ ]:
rl_mro.solve()